In [0]:
# 1. Data Munging -
# 2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
# Apply inferSchema and toDF to create a DF and analyse the actual data.
# Analyse the schema, datatypes, columns etc.,
# Analyse the duplicate records count and summary of the dataframe.

logistics_df=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")

logistics_df.show()
logistics_df.printSchema() # column details
print(logistics_df.dtypes) # column details as list
print(logistics_df.columns) # columns as list
logistics_df.describe().show() # basic summary - count/mean/stddiv/min/max
logistics_df.summary().show() # with percentile details
logistics_df.schema # struct type of column info

#dataframe
from pyspark.sql.window import Window
from pyspark.sql.functions import *

w_spec = Window.partitionBy("shipment_id").orderBy("shipment_id")
final_df = logistics_df.withColumn("row_number",row_number().over(w_spec))
final_df.filter("row_number=2").drop("row_number").show()

#sql
logistics_df.createOrReplaceTempView("logistics")
dup_qry='''select count(*),shipment_id from logistics 
group by shipment_id
having count(*) >1'''
spark.sql(dup_qry).show()

In [0]:
# a. Passive Data Munging - (File: logistics_source1 and logistics_source2)

# shipment_id is non-numericr
# age is not an integer

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")

logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type")

#logistics_df1.show(5)
#logistics_df2.show(5)

joined_df = logistics_df1.join(logistics_df2,logistics_df1.shipment_id==logistics_df2.shipment_id, how="inner").drop(logistics_df2.shipment_id,logistics_df2.first_name,logistics_df2.last_name,logistics_df2.age,logistics_df2.role)

joined_df.show(5)

from pyspark.sql.functions import col

filtered_df = joined_df.\
    filter(~col("shipment_id").cast("string").rlike("^[0-9]+$") \
    & (col("age")).cast("string").isNotNull())

filtered_df.show()



In [0]:
# b. Active Data Munging File: logistics_source1 and logistics_source2

# 1.Combining Data + Schema Merging (Structuring)

'''
both files without enforcing schema
Align them into a single canonical schema: shipment_id, first_name, last_name, age, role, hub_location, vehicle_type, data_source
Add data_source column with values as: system1, system2 in the respective dataframes
''' 

from pyspark.sql.functions import col,lit

logistics_df1=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source1.txt",inferSchema="True",header=True)

logistics1= logistics_df1.withColumn("data_source",lit("source1"))


logistics_df2=spark.read.csv("/Volumes/usecasecatalog/usecaseschema/usecasevolume/logistics_source2.txt",inferSchema=True,header=True)

logistics2= logistics_df2.withColumn("data_source",lit("source2"))

combined_df = logistics1.unionByName(logistics2,allowMissingColumns=True)

# 2. Cleansing, Scrubbing:
'''
Cleansing (removal of unwanted datasets)

Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role
Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name
Join Readiness Rule - Drop records where the join key is null: shipment_id
'''
#Scrubbing (convert raw to tidy)
'''
4. Age Defaulting Rule - Fill NULL values in the age column with: -1
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN
6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1
7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler
'''

column_check_df = combined_df.na.drop(subset=["shipment_id","role"],how="any")

name_complete_df = column_check_df.na.drop(how="all",subset=["first_name","last_name"])

join_readiness_df = name_complete_df.na.drop(subset=["shipment_id"])


scrub1_df1 = join_readiness_df.na.fill(-1, subset=["age"]).\
                  na.fill("Unknown",subset=["vehicle_type"])

scrub1_df1.na.replace({"Truck":"LMV Vehicle","Bike":"Two wheeler"},subset=["vehicle_type"]).show(1000)





